In [1]:
install.packages(c("dplyr","jsonlite","readxl","openxlsx","RSQLite","DBI"), quiet = TRUE)

library(dplyr)
library(jsonlite)
library(readxl)
library(RSQLite)
library(DBI)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [4]:

url <- "https://raw.githubusercontent.com/databricks/Spark-The-Definitive-Guide/master/data/retail-data/all/online-retail-dataset.csv"
download.file(url, destfile = "online_retail_raw.csv", mode = "wb")

raw <- read.csv("online_retail_raw.csv", stringsAsFactors = FALSE)
cat("Rows:", nrow(raw), " Cols:", ncol(raw), "\n")
str(raw)

Rows: 541909  Cols: 8 
'data.frame':	541909 obs. of  8 variables:
 $ InvoiceNo  : chr  "536365" "536365" "536365" "536365" ...
 $ StockCode  : chr  "85123A" "71053" "84406B" "84029G" ...
 $ Description: chr  "WHITE HANGING HEART T-LIGHT HOLDER" "WHITE METAL LANTERN" "CREAM CUPID HEARTS COAT HANGER" "KNITTED UNION FLAG HOT WATER BOTTLE" ...
 $ Quantity   : int  6 6 8 6 6 2 6 6 6 32 ...
 $ InvoiceDate: chr  "12/1/2010 8:26" "12/1/2010 8:26" "12/1/2010 8:26" "12/1/2010 8:26" ...
 $ UnitPrice  : num  2.55 3.39 2.75 3.39 3.39 7.65 4.25 1.85 1.85 1.69 ...
 $ CustomerID : int  17850 17850 17850 17850 17850 17850 17850 17850 17850 13047 ...
 $ Country    : chr  "United Kingdom" "United Kingdom" "United Kingdom" "United Kingdom" ...


In [5]:
library(dplyr)
library(jsonlite)
library(openxlsx)

transactions_raw <- raw[, c("InvoiceNo","StockCode","CustomerID","Quantity","InvoiceDate")]

products_raw <- raw %>%
  filter(!is.na(StockCode), StockCode != "") %>%
  group_by(StockCode) %>%
  summarise(
    Description = names(sort(table(Description), decreasing = TRUE))[1],
    UnitPrice = median(UnitPrice, na.rm = TRUE),
    .groups = "drop"
  )

customers_raw <- raw %>%
  filter(!is.na(CustomerID)) %>%
  group_by(CustomerID) %>%
  summarise(Country = names(sort(table(Country), decreasing = TRUE))[1], .groups = "drop")

cat("transactions_raw:", dim(transactions_raw), "\n")
cat("products_raw:", dim(products_raw), "\n")
cat("customers_raw:", dim(customers_raw), "\n")

transactions_raw: 541909 5 
products_raw: 4070 3 
customers_raw: 4372 2 


In [6]:
cat("Missing values per column (transactions):\n")
print(colSums(is.na(transactions_raw)))

cat("\nMissing values per column (products):\n")
print(colSums(is.na(products_raw)))

cat("\nMissing values per column (customers):\n")
print(colSums(is.na(customers_raw)))

cat("\nDuplicate transaction rows:", sum(duplicated(transactions_raw)), "\n")
cat("Zero/negative quantities:", sum(transactions_raw$Quantity <= 0), "\n")
cat("Zero/negative unit prices:", sum(products_raw$UnitPrice <= 0), "\n")

Missing values per column (transactions):
  InvoiceNo   StockCode  CustomerID    Quantity InvoiceDate 
          0           0      135080           0           0 

Missing values per column (products):
  StockCode Description   UnitPrice 
          0           0           0 

Missing values per column (customers):
CustomerID    Country 
         0          0 

Duplicate transaction rows: 5429 
Zero/negative quantities: 10624 
Zero/negative unit prices: 145 


In [7]:
transactions_clean <- transactions_raw %>%
  distinct() %>%
  filter(!is.na(CustomerID), !is.na(InvoiceDate)) %>%
  filter(Quantity > 0)

products_clean <- products_raw %>%
  filter(!is.na(Description)) %>%
  filter(UnitPrice > 0)

customers_clean <- customers_raw %>%
  filter(!is.na(Country)) %>%
  distinct()

cat("Cleaned dimensions:\n")
cat(" transactions_clean:", dim(transactions_clean), "\n")
cat(" products_clean:", dim(products_clean), "\n")
cat(" customers_clean:", dim(customers_clean), "\n\n")

cat("Rows removed - transactions:", nrow(transactions_raw) - nrow(transactions_clean), "\n")
cat("Rows removed - products:", nrow(products_raw) - nrow(products_clean), "\n")
cat("Rows removed - customers:", nrow(customers_raw) - nrow(customers_clean), "\n")

Cleaned dimensions:
 transactions_clean: 392708 5 
 products_clean: 3925 3 
 customers_clean: 4372 2 

Rows removed - transactions: 149201 
Rows removed - products: 145 
Rows removed - customers: 0 


In [8]:
integrated <- transactions_clean %>%
  inner_join(products_clean, by = "StockCode") %>%
  left_join(customers_clean, by = "CustomerID") %>%
  mutate(Revenue = Quantity * UnitPrice)

cat("Integrated dataset dimensions:", dim(integrated), "\n")

unmatched_products <- setdiff(transactions_clean$StockCode, products_clean$StockCode)
unmatched_customers <- integrated %>% filter(is.na(Country))

cat("Unmatched StockCodes dropped by inner_join:", length(unique(unmatched_products)), "\n")
cat("Transactions with unmatched CustomerID (Country = NA):", nrow(unmatched_customers), "\n")

head(integrated)

Integrated dataset dimensions: 392696 9 
Unmatched StockCodes dropped by inner_join: 10 
Transactions with unmatched CustomerID (Country = NA): 0 


,InvoiceNo,StockCode,CustomerID,Quantity,InvoiceDate,Description,UnitPrice,Country,Revenue
,<chr>,<chr>,<int>,<int>,<chr>,<chr>,<dbl>,<chr>,<dbl>
1,536365,85123A,17850,6,12/1/2010 8:26,WHITE HANGING HEART T-LIGHT HOLDER,2.95,United Kingdom,17.7
2,536365,71053,17850,6,12/1/2010 8:26,WHITE METAL LANTERN,3.75,United Kingdom,22.5
3,536365,84406B,17850,8,12/1/2010 8:26,CREAM CUPID HEARTS COAT HANGER,4.15,United Kingdom,33.2
4,536365,84029G,17850,6,12/1/2010 8:26,KNITTED UNION FLAG HOT WATER BOTTLE,4.25,United Kingdom,25.5
5,536365,84029E,17850,6,12/1/2010 8:26,RED WOOLLY HOTTIE WHITE HEART.,4.25,United Kingdom,25.5
6,536365,22752,17850,2,12/1/2010 8:26,SET 7 BABUSHKA NESTING BOXES,8.50,United Kingdom,17.0


In [9]:
total_revenue <- sum(integrated$Revenue)
cat("Total Sales Revenue:", round(total_revenue, 2), "\n")

Total Sales Revenue: 9676073 


In [10]:
top5_products <- integrated %>%
  group_by(StockCode, Description) %>%
  summarise(Revenue = sum(Revenue), .groups = "drop") %>%
  arrange(desc(Revenue)) %>%
  slice_head(n = 5)

top5_products

StockCode,Description,Revenue
<chr>,<chr>,<dbl>
23843,"PAPER CRAFT , LITTLE BIRDIE",168469.60
22423,REGENCY CAKESTAND 3 TIER,157896.00
85123A,WHITE HANGING HEART T-LIGHT HOLDER,108450.85
23166,MEDIUM CERAMIC TOP STORAGE JAR,97395.00
85099B,JUMBO BAG RED RETROSPOT,95842.24


In [11]:
top5_countries <- integrated %>%
  filter(!is.na(Country)) %>%
  group_by(Country) %>%
  summarise(Revenue = sum(Revenue), .groups = "drop") %>%
  arrange(desc(Revenue)) %>%
  slice_head(n = 5)

top5_countries

Country,Revenue
<chr>,<dbl>
United Kingdom,7982782.1
Netherlands,335470.1
EIRE,287976.4
Germany,235746.4
France,206016.7


In [12]:
customer_value <- integrated %>%
  filter(!is.na(CustomerID)) %>%
  group_by(CustomerID) %>%
  summarise(TotalValue = sum(Revenue), .groups = "drop")

top5_customers <- customer_value %>%
  arrange(desc(TotalValue)) %>%
  slice_head(n = 5)

top5_customers

CustomerID,TotalValue
<int>,<dbl>
18102,401990.8
14646,330004.6
17450,180788.2
16446,168472.5
14911,151986.9


In [13]:
q <- quantile(customer_value$TotalValue, probs = c(0.25, 0.5, 0.75))
print(q)

customer_value <- customer_value %>%
  mutate(Segment = case_when(
    TotalValue <= q[1] ~ "Low Value",
    TotalValue <= q[2] ~ "Medium Value",
    TotalValue <= q[3] ~ "High Value",
    TRUE ~ "Premium"
  ))

table(customer_value$Segment)

    25%     50%     75% 
 312.06  699.43 1738.87 



  High Value    Low Value Medium Value      Premium 
        1084         1085         1085         1085 

In [14]:
best_market <- top5_countries$Country[1]

worst_market <- integrated %>%
  filter(!is.na(Country)) %>%
  group_by(Country) %>%
  summarise(Revenue = sum(Revenue), .groups = "drop") %>%
  arrange(Revenue) %>%
  slice_head(n = 1)

cat("High-performing market:", best_market,
    "- highest total revenue, driven by transaction volume.\n")
cat("Underperforming market:", worst_market$Country,
    "- revenue of", round(worst_market$Revenue, 2),
    "- lowest among all markets.\n")

High-performing market: United Kingdom - highest total revenue, driven by transaction volume.
Underperforming market: Saudi Arabia - revenue of 162.96 - lowest among all markets.


In [15]:
con <- dbConnect(RSQLite::SQLite(), "retail_sales.db")
dbWriteTable(con, "retail_sales", integrated, overwrite = TRUE)

cat("Rows written to retail_sales table:",
    dbGetQuery(con, "SELECT COUNT(*) AS n FROM retail_sales")$n, "\n")

Rows written to retail_sales table: 392696 


In [16]:
query1 <- dbGetQuery(con, "
  SELECT CustomerID, ROUND(SUM(Revenue), 2) AS TotalRevenue
  FROM retail_sales
  WHERE CustomerID IS NOT NULL
  GROUP BY CustomerID
  ORDER BY TotalRevenue DESC
  LIMIT 5;
")
query1

CustomerID,TotalRevenue
<int>,<dbl>
18102,401990.8
14646,330004.6
17450,180788.2
16446,168472.5
14911,151986.9


In [17]:
query2 <- dbGetQuery(con, "
  SELECT Country, ROUND(SUM(Revenue), 2) AS TotalRevenue
  FROM retail_sales
  WHERE Country IS NOT NULL
  GROUP BY Country
  ORDER BY TotalRevenue DESC;
")
query2

Country,TotalRevenue
<chr>,<dbl>
United Kingdom,7982782.15
Netherlands,335470.06
EIRE,287976.43
Germany,235746.39
France,206016.70
Australia,160732.61
Spain,65580.22
Switzerland,57365.78
Belgium,43016.60


In [18]:
dbDisconnect(con)

In [19]:

list.files(pattern = ".db")


[1] "retail_sales.db"